In [21]:
import json
import os
from datetime import datetime, timedelta

from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
MODEL = "claude-haiku-4-5"

WORKSPACE = os.path.abspath("./text_editor_workspace")
os.makedirs(WORKSPACE, exist_ok=True)

In [22]:
# --- Tool Implementations ---

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


def add_duration_to_datetime(datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"):
    date = datetime.strptime(datetime_str, input_format)
    days_per_month = [31, 29 if (lambda y: y % 4 == 0 and (y % 100 != 0 or y % 400 == 0))(date.year) else 28,
                      31, 30, 31, 30, 31, 31, 30, 31, 30, 31]

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12 or 12
        if month == 12:
            year -= 1 if date.month + duration % 12 == 0 else 0
        day = min(date.day, days_per_month[month - 1])
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting reminder for {timestamp}:\n{content}\n----")
    return timestamp


def handle_text_editor(command, path, **kwargs):
    """Client-side handler for Claude's str_replace_based_edit_tool."""
    abs_path = os.path.abspath(os.path.join(WORKSPACE, path.lstrip("/\\")))
    if not abs_path.startswith(WORKSPACE):
        return "Error: path traversal not allowed"

    if command == "view":
        if os.path.isdir(abs_path):
            try:
                entries = os.listdir(abs_path)
                return "\n".join(entries) if entries else "(empty directory)"
            except FileNotFoundError:
                return f"Error: directory not found: {path}"
        try:
            with open(abs_path, "r", encoding="utf-8") as f:
                lines = f.readlines()
            return "".join(f"{i+1}\t{line}" for i, line in enumerate(lines)) or "(empty file)"
        except FileNotFoundError:
            return f"Error: file not found: {path}"

    elif command == "create":
        os.makedirs(os.path.dirname(abs_path), exist_ok=True)
        with open(abs_path, "w", encoding="utf-8") as f:
            f.write(kwargs.get("file_text", ""))
        return f"Created {path}"

    elif command == "str_replace":
        old_str, new_str = kwargs.get("old_str", ""), kwargs.get("new_str", "")
        try:
            with open(abs_path, "r", encoding="utf-8") as f:
                content = f.read()
        except FileNotFoundError:
            return f"Error: file not found: {path}"
        count = content.count(old_str)
        if count == 0:
            return "Error: old_str not found in file"
        if count > 1:
            return f"Error: old_str matches {count} locations — must be unique"
        with open(abs_path, "w", encoding="utf-8") as f:
            f.write(content.replace(old_str, new_str, 1))
        return "Edit applied"

    elif command == "insert":
        insert_line = kwargs.get("insert_line", 0)
        new_str = kwargs.get("new_str", "")
        try:
            with open(abs_path, "r", encoding="utf-8") as f:
                lines = f.readlines()
        except FileNotFoundError:
            return f"Error: file not found: {path}"
        lines.insert(insert_line, new_str if new_str.endswith("\n") else new_str + "\n")
        with open(abs_path, "w", encoding="utf-8") as f:
            f.writelines(lines)
        return f"Inserted after line {insert_line}"

    return f"Error: unknown command '{command}'"

In [23]:
# --- Tool Schemas ---

get_current_datetime_schema = {
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format string.",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A strftime format string (e.g. '%Y-%m-%d'). Defaults to '%Y-%m-%d %H:%M:%S'.",
                "default": "%Y-%m-%d %H:%M:%S",
            }
        },
        "required": [],
    },
}

add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": (
        "Adds a specified duration to a datetime string and returns the resulting datetime. "
        "Supports seconds, minutes, hours, days, weeks, months, and years. "
        "Output format: 'Thursday, April 03, 2025 10:30:00 AM'."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "Input datetime string, parsed according to input_format.",
            },
            "duration": {
                "type": "number",
                "description": "Amount of time to add. Use negative values for past dates. Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "Time unit: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "strptime format for parsing datetime_str (e.g. '%Y-%m-%d'). Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Schedules a reminder notification to be delivered to the user at the specified timestamp.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The reminder message text (e.g. 'Take medication', 'Join standup call').",
            },
            "timestamp": {
                "type": "string",
                "description": "ISO 8601 timestamp when the reminder fires (YYYY-MM-DDTHH:MM:SS).",
            },
        },
        "required": ["content", "timestamp"],
    },
}

# All tools available to the assistant. Claude decides which (if any) to invoke
# based on the user's message — callers never need to choose or pass these.
ALL_TOOLS = [
    get_current_datetime_schema,
    add_duration_to_datetime_schema,
    set_reminder_schema,
    {"type": "text_editor_20250728", "name": "str_replace_based_edit_tool"},
]

In [24]:
# --- Tool Dispatcher ---

def run_tool(tool_name, tool_input):
    handlers = {
        "get_current_datetime": get_current_datetime,
        "add_duration_to_datetime": add_duration_to_datetime,
        "set_reminder": set_reminder,
        "str_replace_based_edit_tool": handle_text_editor,
    }
    if tool_name not in handlers:
        raise ValueError(f"Unknown tool: {tool_name}")
    return handlers[tool_name](**tool_input)


def run_tools(message):
    tool_results = []
    for block in message.content:
        if block.type != "tool_use":
            continue
        print(f"\n[Tool: {block.name}] {block.input}")
        try:
            output = run_tool(block.name, block.input)
            print(f"  → {output}")
            result = {"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(output), "is_error": False}
        except Exception as e:
            print(f"  → Error: {e}")
            result = {"type": "tool_result", "tool_use_id": block.id, "content": f"Error: {e}", "is_error": True}
        tool_results.append(result)
    return tool_results

In [25]:
# --- Chat Utilities ---

def text_from_message(message):
    return "\n".join(block.text for block in message.content if block.type == "text")

In [26]:
# --- Conversation Runner ---

def run_conversation(messages):
    """
    Agentic loop. Callers only provide messages — tool selection is handled internally.
    Claude decides which tools (if any) to call based on the user's message.
    """
    while True:
        response = client.messages.create(
            model=MODEL,
            max_tokens=4096,
            tools=ALL_TOOLS,
            messages=messages,
        )

        for block in response.content:
            if hasattr(block, "text"):
                print(block.text)

        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            break

        messages.append({"role": "user", "content": run_tools(response)})

    return messages

In [27]:
# --- Example: Reminder ---
# The caller only sends a message. No tools, no model — just intent.

messages = [{"role": "user", "content": "Set a reminder for my doctor's appointment. It's 177 days after Jan 1st, 2050."}]
run_conversation(messages)

I'll help you set a reminder for your doctor's appointment. Let me first calculate what date is 177 days after January 1st, 2050.

[Tool: add_duration_to_datetime] {'datetime_str': '2050-01-01', 'duration': 177, 'unit': 'days'}
  → Monday, June 27, 2050 12:00:00 AM
Now I'll set a reminder for your doctor's appointment on June 27, 2050:

[Tool: set_reminder] {'content': "Doctor's appointment", 'timestamp': '2050-06-27T12:00:00'}
----
Setting reminder for 2050-06-27T12:00:00:
Doctor's appointment
----
  → 2050-06-27T12:00:00
Perfect! I've set a reminder for your doctor's appointment on **Monday, June 27, 2050** at 12:00 AM.


[{'role': 'user',
  'content': "Set a reminder for my doctor's appointment. It's 177 days after Jan 1st, 2050."},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll help you set a reminder for your doctor's appointment. Let me first calculate what date is 177 days after January 1st, 2050.", type='text'),
   ToolUseBlock(id='toolu_01TdqtTZAoRdv6HxJARAcZJq', caller=DirectCaller(type='direct'), input={'datetime_str': '2050-01-01', 'duration': 177, 'unit': 'days'}, name='add_duration_to_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01TdqtTZAoRdv6HxJARAcZJq',
    'content': '"Monday, June 27, 2050 12:00:00 AM"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="Now I'll set a reminder for your doctor's appointment on June 27, 2050:", type='text'),
   ToolUseBlock(id='toolu_01TC6mzZSCFdyVzRV5rwLQ98', caller=DirectCaller(type='direct'), input={'content': "Doct

## Built-in Text Editor Tool (`text_editor_20250728`)

Claude's built-in text editor lets you create and edit files via natural language prompts.
The tool uses `type: "text_editor_20250728"` with **no** `input_schema` — it's Anthropic-defined.

| Command | Description |
|---|---|
| `view` | Display a file (with line numbers) or list a directory |
| `create` | Write a new file with the given content |
| `str_replace` | Replace a unique string in a file |
| `insert` | Insert text after a specific line number |

All file operations are sandboxed to the `WORKSPACE` directory.

In [28]:
# --- Example: Text Editor ---
# Same run_conversation — Claude picks the text editor tool based on the message.

messages = [{"role": "user", "content": "Create a Python file called hello.py that prints hello world"}]
run_conversation(messages)


[Tool: str_replace_based_edit_tool] {'command': 'create', 'path': '/hello.py', 'file_text': 'print("Hello World")\n'}
  → Created /hello.py
Done! I've created a Python file called `hello.py` that prints "Hello World". You can run it with the command:
```bash
python hello.py
```


[{'role': 'user',
  'content': 'Create a Python file called hello.py that prints hello world'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_017uanocPD1JBAXAnUdmJq1B', caller=DirectCaller(type='direct'), input={'command': 'create', 'path': '/hello.py', 'file_text': 'print("Hello World")\n'}, name='str_replace_based_edit_tool', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_017uanocPD1JBAXAnUdmJq1B',
    'content': '"Created /hello.py"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='Done! I\'ve created a Python file called `hello.py` that prints "Hello World". You can run it with the command:\n```bash\npython hello.py\n```', type='text')]}]